# 07주차: PyTorch CIFAR-10 모델 구조와 데이터 증강

## 학습 목표
CIFAR-10 컬러 이미지에 6주차와 동일한 `SmallCNN` 구조를 입력 채널만 3으로 바꾸어 그대로 적용해 보고, 더 깊은 `ImprovedCNN` 구조와 데이터 증강이 검증·테스트 성능에 어떤 영향을 주는지 비교합니다. 같은 데이터 분할과 같은 학습 설정(손실 함수, 옵티마이저, **에포크 수**)을 유지한 채 모델 구조와 데이터 전처리만 바꾸어, 성능 변화의 원인을 하나씩 구분해서 관찰하는 실험 설계를 연습합니다.

Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 세 실험을 모두 합쳐 T4 기준 15분 내외가 걸립니다. GPU가 없어도 CPU에서 실행되지만 훨씬 오래 걸리므로 반드시 T4를 켜세요. CIFAR-10은 torchvision이 자동으로 내려받으므로 Google Drive 연결, Drive 마운트, 파일 업로드는 필요하지 않습니다.

## 관찰 질문
- 6주차의 `SmallCNN` 구조를 그대로 쓰는데 입력 채널만 3으로 바뀌면 어떤 층의 파라미터 수가 달라질까요?
- Fashion-MNIST의 흑백 의류 이미지와 달리, CIFAR-10의 컬러 사물 이미지는 왜 더 어려운 분류 문제일까요?
- 데이터 증강은 학습 정확도와 검증 정확도의 차이(과적합 정도)를 어떻게 바꿀까요?
- 데이터 증강의 효과는 왜 학습 초반이 아니라 **학습이 충분히 진행된 뒤에야** 드러날까요?


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

full_train_base = datasets.CIFAR10(root="./data", train=True, download=True, transform=base_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=base_transform)

split_generator = torch.Generator().manual_seed(42)
permutation = torch.randperm(len(full_train_base), generator=split_generator).tolist()
train_indices = permutation[:45000]
val_indices = permutation[45000:]
print("분할 크기(훈련/검증):", len(train_indices), len(val_indices))

train_dataset = Subset(full_train_base, train_indices)
val_dataset = Subset(full_train_base, val_indices)

batch_size = 128
loader_options = {"batch_size": batch_size, "num_workers": 2, "pin_memory": torch.cuda.is_available()}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
print("데이터 수:", len(train_dataset), len(val_dataset), len(test_dataset))
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]


## 6주차 SmallCNN 구조 재사용
아래 `SmallCNN`은 6주차 노트북의 클래스 정의를 한 글자도 바꾸지 않고 그대로 옮긴 것입니다. 유일한 차이는 인스턴스를 만들 때 `SmallCNN(in_channels=3)`으로 채널 수만 지정한다는 점입니다. 첫 번째 합성곱 층 `nn.Conv2d(in_channels, 16, 3, padding=1)`의 입력 채널만 1에서 3으로 바뀌므로, 그 층의 파라미터 수만 늘어나고 나머지 구조는 완전히 동일합니다.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

sample_images, sample_labels = next(iter(train_loader))
with torch.no_grad():
    shape_logits = SmallCNN(in_channels=3).to(device)(sample_images.to(device))
print("입력 텐서 shape:", sample_images.shape)
print("출력 텐서 shape:", shape_logits.shape)


## 공통 학습·평가 함수
`train_one_epoch`, `evaluate`, `fit`, `plot_history`, `count_parameters`는 6주차와 같은 역할을 합니다. 이 함수들을 세 가지 실험(기준 모델, 구조 개선 모델, 데이터 증강 모델)에서 그대로 재사용해, 모델 구조와 데이터 조건만 바뀌고 학습 절차 자체는 바뀌지 않도록 합니다.

여기에 두 가지를 추가합니다. `overfitting_gap`은 마지막 에포크의 **학습 정확도 − 검증 정확도**를 계산합니다. 이 값이 클수록 모델이 학습 데이터에만 맞춰졌다는 뜻이므로, 증강의 효과를 정확도뿐 아니라 과적합 정도로도 볼 수 있습니다. `EPOCHS`는 세 실험이 공유하는 에포크 수입니다. 에포크 수까지 똑같이 맞춰야 "구조를 바꾼 효과"와 "증강을 넣은 효과"를 다른 요인과 섞이지 않게 분리할 수 있습니다.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for batch_images, batch_labels in loader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * batch_labels.size(0)
        correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            logits = model(batch_images)
            loss_sum += criterion(logits, batch_labels).item() * batch_labels.size(0)
            correct += (logits.argmax(dim=1) == batch_labels).sum().item()
            total += batch_labels.size(0)
    return loss_sum / total, 100 * correct / total

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
        for key, value in [("train_loss", train_loss), ("train_acc", train_accuracy), ("val_loss", val_loss), ("val_acc", val_accuracy)]:
            history[key].append(value)
        print(f"Epoch {epoch}/{epochs}: train={train_accuracy:.2f}%, val={val_accuracy:.2f}%, 격차={train_accuracy - val_accuracy:+.2f}%p")
    return history

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].legend()
    plt.show()

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def overfitting_gap(history):
    """마지막 에포크의 학습 정확도 - 검증 정확도. 값이 클수록 과적합이 심합니다."""
    return history["train_acc"][-1] - history["val_acc"][-1]

# 세 실험 모두 같은 에포크 수를 사용해, 성능 차이가 "구조"와 "데이터 증강"에서만 오도록 합니다.
# 22로 잡은 이유: 8에포크에서는 증강 없는 모델조차 아직 과적합에 들어가지 않아
# (학습-검증 격차가 1%p 미만) 증강이 줄여 줄 과적합 자체가 없습니다.
# 20에포크를 넘겨야 증강 없는 모델의 격차가 10%p 이상으로 벌어지면서
# 증강의 효과가 검증·테스트 정확도에서 뚜렷하게 드러납니다.
EPOCHS = 22
criterion = nn.CrossEntropyLoss()
print("모든 실험 공통 에포크 수:", EPOCHS)


## 기준 모델(SmallCNN) 학습
`CrossEntropyLoss`와 `Adam(lr=1e-3)`으로 `SmallCNN(in_channels=3)`을 `EPOCHS`만큼 학습해 `baseline_history`를 만듭니다. 뒤의 두 실험과 에포크 수를 똑같이 맞추므로, 이 모델과의 성능 차이는 "학습을 덜 했기 때문"이 아니라 "구조가 작기 때문"이라고 말할 수 있습니다.

### 관찰 질문
- 6주차 Fashion-MNIST에서는 짧은 학습만으로도 검증 정확도가 꽤 높게 올라갔습니다. 같은 구조의 모델이 CIFAR-10에서는 왜 검증 정확도가 낮은 수준에 머무를까요?
- 학습 정확도와 검증 정확도의 격차가 커진다면 무엇을 의심해 볼 수 있을까요?


In [ ]:
seed_everything(42)
model_baseline = SmallCNN(in_channels=3).to(device)
optimizer_baseline = torch.optim.Adam(model_baseline.parameters(), lr=1e-3)
baseline_params = count_parameters(model_baseline)
print("기준 모델(SmallCNN) 파라미터 수:", baseline_params)
baseline_history = fit(model_baseline, train_loader, val_loader, criterion, optimizer_baseline, device, epochs=EPOCHS)
plot_history(baseline_history, "Baseline SmallCNN")
baseline_test_loss, baseline_test_accuracy = evaluate(model_baseline, test_loader, criterion, device)
print(f"기준 모델 테스트 정확도: {baseline_test_accuracy:.2f}%")
print(f"기준 모델 과적합 격차(학습-검증): {overfitting_gap(baseline_history):+.2f}%p")


## 구조 개선 모델(ImprovedCNN) 학습
`ImprovedCNN`은 합성곱 블록을 세 단계(32, 64, 128 채널)로 늘리고, 블록마다 합성곱을 두 번 반복한 뒤 풀링합니다. `use_batchnorm`과 `dropout` 인자는 이번 실험에서 각각 `False`와 `0.0`으로 두어, 배치 정규화나 드롭아웃 없이 순수하게 "더 깊고 넓은 구조" 하나만 바꾼 효과를 관찰합니다. 학습 설정(손실 함수, 옵티마이저, 에포크 수, 데이터)은 기준 모델과 동일하게 유지합니다.

### 관찰 질문
- 에포크가 진행될수록 학습 정확도와 검증 정확도의 격차(`격차` 출력)는 어떻게 변하나요? 몇 에포크쯤부터 벌어지기 시작하나요?
- 검증 정확도가 더 이상 오르지 않는데 학습 정확도만 계속 오른다면, 모델은 무엇을 배우고 있는 걸까요?


In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, use_batchnorm=False, dropout=0.0):
        super().__init__()
        def block(in_ch, out_ch):
            layers = [nn.Conv2d(in_ch, out_ch, 3, padding=1)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers += [nn.ReLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            return layers
        self.features = nn.Sequential(
            *block(3, 32), *block(32, 64), *block(64, 128),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

seed_everything(42)
improved_model = ImprovedCNN().to(device)
optimizer_improved = torch.optim.Adam(improved_model.parameters(), lr=1e-3)
improved_params = count_parameters(improved_model)
print("개선 모델(ImprovedCNN) 파라미터 수:", improved_params)
improved_history = fit(improved_model, train_loader, val_loader, criterion, optimizer_improved, device, epochs=EPOCHS)
plot_history(improved_history, "Improved CNN")
improved_test_loss, improved_test_accuracy = evaluate(improved_model, test_loader, criterion, device)
print(f"개선 모델 테스트 정확도: {improved_test_accuracy:.2f}%")
print(f"개선 모델 과적합 격차(학습-검증): {overfitting_gap(improved_history):+.2f}%p")


## 데이터 증강
`RandomCrop(32, padding=4)`와 `RandomHorizontalFlip()`을 훈련 데이터에만 적용합니다. 검증·테스트 데이터는 기준 전처리(`base_transform`)를 그대로 사용해, 증강이 "평가 방식"이 아니라 "훈련 방식"만 바꾼다는 점을 유지합니다. 같은 분할 인덱스(`train_indices`), 같은 `ImprovedCNN` 구조, 같은 옵티마이저 설정, 같은 에포크 수(`EPOCHS`)로 학습하므로 성능 차이는 오직 데이터 증강 여부에서만 생깁니다.

증강은 매 에포크 같은 이미지를 조금씩 다르게 잘라내고 뒤집어 보여줍니다. 모델 입장에서는 매번 처음 보는 이미지가 오는 셈이라 **훈련 데이터를 통째로 외우기가 어려워집니다**. 아래 표본 그림에서 같은 클래스의 이미지가 위치나 좌우 방향이 조금씩 어긋나 있는 것을 확인하세요.


In [ ]:
augment_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

full_train_augment = datasets.CIFAR10(root="./data", train=True, download=True, transform=augment_transform)
augmented_train_dataset = Subset(full_train_augment, train_indices)
augmented_train_loader = DataLoader(augmented_train_dataset, shuffle=True, **loader_options)

def denormalize(tensor):
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

augment_sample_images, augment_sample_labels = next(iter(augmented_train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for image, label, axis in zip(augment_sample_images[:10], augment_sample_labels[:10], axes.flat):
    axis.imshow(denormalize(image).permute(1, 2, 0))
    axis.set_title(class_names[label.item()])
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
seed_everything(42)
augmented_model = ImprovedCNN().to(device)
optimizer_augmented = torch.optim.Adam(augmented_model.parameters(), lr=1e-3)
augmented_params = count_parameters(augmented_model)
print("증강 모델(ImprovedCNN + 증강) 파라미터 수:", augmented_params)
augmented_history = fit(augmented_model, augmented_train_loader, val_loader, criterion, optimizer_augmented, device, epochs=EPOCHS)
plot_history(augmented_history, "Augmented ImprovedCNN")
augmented_test_loss, augmented_test_accuracy = evaluate(augmented_model, test_loader, criterion, device)
print(f"증강 모델 테스트 정확도: {augmented_test_accuracy:.2f}%")
print(f"증강 모델 과적합 격차(학습-검증): {overfitting_gap(augmented_history):+.2f}%p")


## 세 결과 비교와 학생 활동
기준 모델, 구조 개선 모델, 데이터 증강 모델의 파라미터 수와 정확도, 그리고 **과적합 격차**를 표와 그래프로 비교합니다. 세 실험은 데이터 분할·옵티마이저·학습률·에포크 수가 모두 같으므로, 차이는 오직 "구조"와 "증강" 두 가지에서만 나옵니다.

세 번째 그래프(Overfitting Gap)를 특히 눈여겨보세요. 증강 없는 개선 모델의 격차는 에포크가 갈수록 계속 벌어지지만, 증강 모델의 격차는 낮게 유지됩니다. 검증 정확도가 정체되는데 학습 정확도만 오르는 구간이 바로 모델이 훈련 데이터를 외우기 시작한 지점입니다.

### 학생 활동
1. 세 모델의 파라미터 수를 비교하고, 파라미터가 많다고 항상 정확도가 좋아지는지 확인하세요.
2. `EPOCHS`를 8로 줄여서 다시 실행하면 증강 모델이 개선 모델을 **이기지 못합니다**. 왜 그런지, 과적합 격차 그래프를 근거로 설명해 보세요. (힌트: 8에포크 시점의 개선 모델 격차는 몇 %p인가요?)
3. `augment_transform`에서 `RandomHorizontalFlip()`만 제거하거나 `RandomCrop`만 제거해 다시 학습하고 어떤 증강이 더 중요한지 비교하세요.
4. `ImprovedCNN(use_batchnorm=True)`로 바꾸어 학습하면 오늘 결과와 어떻게 달라질지 예상하고 실제로 확인하세요.


In [ ]:
comparison_rows = [
    ("Baseline SmallCNN", baseline_params, baseline_history, baseline_test_accuracy),
    ("Improved CNN", improved_params, improved_history, improved_test_accuracy),
    ("Augmented ImprovedCNN", augmented_params, augmented_history, augmented_test_accuracy),
]
header = f"{'모델':<24}{'파라미터 수':>13}{'최종 검증acc(%)':>15}{'최고 검증acc(%)':>15}{'테스트acc(%)':>13}{'과적합 격차(%p)':>16}"
print(header)
print("-" * 100)
for name, params, history, test_acc in comparison_rows:
    print(f"{name:<24}{params:>13,}{history['val_acc'][-1]:>15.2f}"
          f"{max(history['val_acc']):>15.2f}{test_acc:>13.2f}{overfitting_gap(history):>+16.2f}")

print()
print(f"구조 개선 효과 (Improved - Baseline): 테스트 {improved_test_accuracy - baseline_test_accuracy:+.2f}%p")
print(f"데이터 증강 효과 (Augmented - Improved): 테스트 {augmented_test_accuracy - improved_test_accuracy:+.2f}%p, "
      f"과적합 격차 {overfitting_gap(augmented_history) - overfitting_gap(improved_history):+.2f}%p")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for history, label in [
    (baseline_history, "baseline"),
    (improved_history, "improved"),
    (augmented_history, "augmented"),
]:
    epochs_axis = range(1, len(history["val_acc"]) + 1)
    axes[0].plot(epochs_axis, history["val_loss"], label=label)
    axes[1].plot(epochs_axis, history["val_acc"], label=label)
    gaps = [t - v for t, v in zip(history["train_acc"], history["val_acc"])]
    axes[2].plot(epochs_axis, gaps, label=label)
axes[0].set_title("Validation Loss")
axes[1].set_title("Validation Accuracy")
axes[2].set_title("Overfitting Gap (train acc - val acc)")
axes[2].axhline(0, color="gray", linewidth=0.8, linestyle="--")
for axis in axes:
    axis.set_xlabel("epoch")
    axis.legend()
plt.tight_layout()
plt.show()


## 최고 성능 모델의 오분류 확인
세 모델 중 가장 성능이 좋은 `augmented_model`을 테스트셋에 적용해 오분류 이미지를 최대 10개까지 모아봅니다. 6주차와 같은 방식으로 예측과 정답을 함께 표시하되, 증강 학습에는 정규화된 텐서를 사용하므로 화면에 그리기 전에 `denormalize`로 픽셀 값을 되돌립니다.


In [ ]:
augmented_model.eval()
mistake_images, mistake_predictions, mistake_labels = [], [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_predictions = augmented_model(batch_images.to(device)).argmax(dim=1).cpu()
        wrong = batch_predictions != batch_labels
        for image, prediction, label in zip(batch_images[wrong], batch_predictions[wrong], batch_labels[wrong]):
            mistake_images.append(image)
            mistake_predictions.append(prediction)
            mistake_labels.append(label)
            if len(mistake_images) == 10:
                break
        if len(mistake_images) == 10:
            break

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, image, prediction, label in zip(axes.flat, mistake_images, mistake_predictions, mistake_labels):
    axis.imshow(denormalize(image).permute(1, 2, 0))
    axis.set_title(f"Pred: {class_names[prediction.item()]}\nActual: {class_names[label.item()]}")
    axis.axis("off")
for axis in axes.flat[len(mistake_images):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 핵심 정리와 8주차 연결

### 핵심 정리
- 6주차의 `SmallCNN`을 채널 수만 바꿔 그대로 CIFAR-10에 적용하면, 흑백 의류보다 복잡한 컬러 사물 이미지 앞에서 모델 용량의 한계가 뚜렷이 드러났습니다. 세 실험의 에포크 수를 똑같이 맞췄으므로, 이 차이는 학습량이 아니라 구조에서 온 것입니다.
- 합성곱 블록을 깊고 넓게 만든 `ImprovedCNN`은 같은 데이터·같은 학습 설정에서도 기준 모델보다 높은 정확도를 보여, 구조 자체의 표현력이 성능에 미치는 영향을 확인했습니다. 다만 학습이 진행될수록 학습 정확도만 계속 오르고 검증 정확도는 정체되는 **과적합**이 나타났습니다.
- `RandomCrop`과 `RandomHorizontalFlip`으로 학습 데이터만 증강한 `augmented_model`은 같은 `ImprovedCNN` 구조에서 과적합 격차를 크게 줄이고, 그 결과 검증·테스트 정확도도 더 높였습니다.
- **증강의 효과는 과적합이 시작된 뒤에야 나타납니다.** 학습 초반에는 증강이 오히려 학습 속도를 늦춰 정확도가 낮게 보입니다. 아직 외울 만큼 학습하지도 않은 모델에게 "외우지 말라"고 하는 규제는 손해일 뿐이기 때문입니다. 규제 기법의 효과를 판단할 때는 반드시 충분히 학습한 뒤에 비교해야 한다는 점을 기억하세요.
- 오분류 이미지들은 여전히 모양이나 배경이 비슷한 클래스끼리 헷갈리는 경향을 보여줍니다.

### 8주차 연결 질문
오늘 세 모델은 모두 같은 학습률(`1e-3`)과 같은 옵티마이저(Adam)를 사용했습니다. 8주차에서는 학습률과 옵티마이저(SGD·Adam) 자체를 짧게 비교하고, `ImprovedCNN`에 배치 정규화와 드롭아웃을 더했을 때 학습 안정성과 과적합 방지에 어떤 차이가 생기는지, 그리고 Early Stopping이 왜 마지막 에포크가 아니라 가장 좋았던 가중치를 선택하는지 살펴봅니다. 오늘 관찰한 과적합 격차 그래프를 떠올리며, 배치 정규화·드롭아웃이 그 격차를 어떻게 줄여줄지 미리 예상해 보세요. 또 오늘 "증강 없는 모델은 검증 정확도가 정체된 뒤에도 계속 학습했다"는 점이, 8주차의 Early Stopping이 필요한 이유와 어떻게 이어지는지도 생각해 보세요.
